# EVE 310 - Lab 09: Batch processing many files

**Module 3 | 10/22/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab09-batch-processing/notebooks/lab09-tutorial.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Read one data file, name it from its file name, and run a cleaning-and-plotting analysis on it
2. List every file in a folder and keep only the CSV files, with a loop that builds a list
3. Repeat the same analysis on every file with one for loop, saving one figure per file

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It downloads the 24 water files and a notes file into `DATA_DIR` and creates `FIGURES_DIR`.
3. **This tutorial is typed, not read.** Most code cells are missing one or two lines, marked `--`. In lab, type them along with the instructor, then run the cell. The text above each cell says what the lines do, and **Show the finished cell** below it has the answer if you get stuck or are working on your own. A cell with a `--` still in it cannot run, so **Run all** fails until every blank is filled.
4. Nothing to submit for this notebook. The **activity** notebook is the one you download as `.ipynb` and upload to Gradescope (optional, for feedback).

### Tutorial

So far every lab has analyzed **one file at a time**: load it, clean it, plot it, fit it. Real data rarely arrives as one file. A facilities office has one meter file per building, a monitoring network has one file per sensor, a lab has one file per sample. Today's job: 24 files of daily water consumption, one per campus building, and the same analysis wanted for each of them, an outlier clean and a box plot by month. Doing it 24 times by hand means 24 copies of the same cells and 24 chances to forget one. Doing it with a for loop means one copy of the code and one click.

**Nothing new from lecture today.** The pieces are all from earlier labs: `pd.to_datetime` and a box plot (Lab 4), the 3 standard deviation outlier rule (Lab 6), and a for loop with an if inside it (Labs 7 and 8). Two small things are new in Python: `os.listdir`, which lists the files in a folder, and a **list** that grows with `.append`, which is Lab 7's counter with names instead of a number.

The plan has two halves. First, the analysis for **one** file, written so that only the file name would change for another file. Then the loop: list the files, and run that same analysis once per file name.

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab09-batch-processing"
BUILDINGS = "AHG AND ARC ART BAT BEN BHD BLD BMA BME BUR CAL CBA CDL CLA CMA CPE CRD CRH DCP DFA EAS ECj EPS".split()
DATA_FILES = ["water_" + code + ".csv" for code in BUILDINGS] + ["water_notes.txt"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**Run as is:** the usual imports plus `os`, Python's module for talking to the operating system. Today it does one thing: list the files in a folder.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. One file: read it and name it

The files are named `water_XXX.csv`, where `XXX` is the three-letter building code. Start with one building, DCP, and keep the file name in a variable, `file_name`, because that is the one thing that will change later.

Two lines to type. The first reads the file from `DATA_DIR`, as in every lab since Lab 3: `pd.read_csv(DATA_DIR / file_name)`. The second pulls the building code out of the file name. A string can be sliced like an array (Lab 2): `file_name[6:9]` is characters 6, 7 and 8, which in `water_DCP.csv` are the `DCP` (count from 0: `w` is 0, the `_` is 5, so the code starts at 6, and the slice stops before 9). The code goes into the plot title and the saved file's name, so it has to come from the file name, not be typed by hand.

**Type it:** `water_df = pd.read_csv(DATA_DIR / file_name)` and `building = file_name[6:9]`.

In [ ]:
file_name = 'water_DCP.csv'

# Read this one file from DATA_DIR into a dataframe
--
# The building code: characters 6, 7 and 8 of the file name
--

print(building)
print(len(water_df), 'days')
water_df.head()

<details><summary>Show the finished cell</summary>

```python
file_name = 'water_DCP.csv'

# Read this one file from DATA_DIR into a dataframe
water_df = pd.read_csv(DATA_DIR / file_name)
# The building code: characters 6, 7 and 8 of the file name
building = file_name[6:9]

print(building)
print(len(water_df), 'days')
water_df.head()
```

</details>

## 2. Dates and months

`DateTime` arrives as text. `pd.to_datetime` turns it into real dates (Lab 4), and once it is a date, `.dt.month` gives the month number of every row, 1 to 12. That number is what the box plot groups by.

**Type it:** `water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])`, then `water_df['Month'] = water_df['DateTime'].dt.month`.

In [ ]:
# Turn the DateTime column into real dates, then pull out the month number
--
--

water_df.head()

<details><summary>Show the finished cell</summary>

```python
# Turn the DateTime column into real dates, then pull out the month number
water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])
water_df['Month'] = water_df['DateTime'].dt.month

water_df.head()
```

</details>

## 3. Remove the outliers

Lab 6's rule: a day is an outlier when its value is more than 3 standard deviations from the mean. The mean, the standard deviation (`ddof=1`, as in Lab 6) and the two limits are given. The column name has spaces and parentheses, so it lives in `col` and is never retyped.

**Type it:** the line that keeps only the rows between the limits, `water_df = water_df[(water_df[col] < upper) & (water_df[col] > lower)]`. Each comparison in its own parentheses, joined with `&` (Lab 8).

In [ ]:
col = 'Water ( Gallons )'

w_mean = np.mean(water_df[col])
w_std = np.std(water_df[col], ddof=1)
upper = w_mean + 3 * w_std
lower = w_mean - 3 * w_std

n_before = len(water_df)
# Keep only the rows between the two limits (Lab 6)
--
print('Removed', n_before - len(water_df), 'outliers;', len(water_df), 'days remain')

<details><summary>Show the finished cell</summary>

```python
col = 'Water ( Gallons )'

w_mean = np.mean(water_df[col])
w_std = np.std(water_df[col], ddof=1)
upper = w_mean + 3 * w_std
lower = w_mean - 3 * w_std

n_before = len(water_df)
# Keep only the rows between the two limits (Lab 6)
water_df = water_df[(water_df[col] < upper) & (water_df[col] > lower)]
print('Removed', n_before - len(water_df), 'outliers;', len(water_df), 'days remain')
```

</details>

57 of DCP's 2,930 days are more than 3 standard deviations from the mean. A meter that reads 17,100 gallons on a day when the building usually uses 800 is a meter problem or a one-off event, and either way it would stretch the box plot's axis until the boxes are flat.

## 4. The monthly box plot

Lab 4 drew one box for one column. pandas can draw one box **per group**: `water_df.boxplot(column=col, by='Month', ax=ax, grid=False)` groups the rows by the `Month` column and draws twelve boxes on the axes `ax`. pandas adds a title of its own above the figure, so `fig.suptitle('')` clears it and `ax.set_title` sets ours, with the building code in it.

The saved file's name also carries the building code: `building + '_water_boxplot.png'` joins the two strings with `+`, giving `DCP_water_boxplot.png`. In section 8 this line runs 24 times; if the name were fixed, every pass would overwrite the one before and only the last building's figure would survive.

**Type it:** the boxplot line, and the `fig.savefig(FIGURES_DIR / (building + '_water_boxplot.png'), dpi=150)` line. Keep the parentheses around the name: `/` runs before `+` otherwise.

In [ ]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots()
# One box per month, drawn on ax
--
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_xlabel('Month')
ax.set_ylabel('Water consumption (gallons per day)')
ax.set_title(building + ' water consumption by month')
fig.suptitle('')
# Save it under a name that says which building it is
--

<details><summary>Show the finished cell</summary>

```python
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots()
# One box per month, drawn on ax
water_df.boxplot(column=col, by='Month', ax=ax, grid=False)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_xlabel('Month')
ax.set_ylabel('Water consumption (gallons per day)')
ax.set_title(building + ' water consumption by month')
fig.suptitle('')
# Save it under a name that says which building it is
fig.savefig(FIGURES_DIR / (building + '_water_boxplot.png'), dpi=150)
```

</details>

DCP uses the most water in late spring and early summer and the least in winter. Open the Files pane on the left (folder icon), then `figures`, and the PNG is there.

## 5. What would change for another file?

Look back over sections 1 to 4. For building ARC instead of DCP, exactly one thing changes: `file_name = 'water_ARC.csv'`. Everything after it, the building code, the dates, the outlier limits, the plot title, the saved file's name, is computed **from** `file_name` or from the data. That is the whole trick of batch processing: write the analysis once so that only the input name changes, then let a loop change the name.

## 6. List the folder

`os.listdir(DATA_DIR)` returns the names of everything in the data folder as a **list** of strings, the Python type from Lab 2 that holds anything, not only numbers. The order it comes back in depends on the computer, so `sorted(...)` puts the names in alphabetical order, which makes the loop in section 8 visit the buildings in the same order on every machine.

**Type it:** `all_files = sorted(os.listdir(DATA_DIR))`.

In [ ]:
# Every file in the data folder, in alphabetical order
--
print(len(all_files), 'files')
print(all_files)

<details><summary>Show the finished cell</summary>

```python
# Every file in the data folder, in alphabetical order
all_files = sorted(os.listdir(DATA_DIR))
print(len(all_files), 'files')
print(all_files)
```

</details>

25 names, not 24: the folder also holds `water_notes.txt`, a text file describing the data. `pd.read_csv` on it does not stop with a clear error; it reads the text as a one-column table, and the crash comes later, with `KeyError: 'DateTime'`, when the loop looks for a column that is not there. Any real folder has such files (a readme, a spreadsheet, a figure), so the loop must only visit the CSVs.

## 7. Keep only the CSVs: a list and `append`

Lab 7 counted values past a threshold with a counter: start at 0, loop, `if` the value passes, add 1. The same pattern **collects** names instead of counting them: start with an empty list `[]`, loop over `all_files`, and `if` the name ends in `.csv`, `append` it to the list. `csv_files.append(name)` adds one item to the end of `csv_files`, so the list grows by one each time the if is true, the way the counter grew by one. Note the dot: `append` belongs to the list it adds to.

`all_files[i].endswith('.csv')` asks the string a yes/no question, True for `water_DCP.csv` and False for `water_notes.txt`, so it goes straight after `if`.

**Type it:** the `if` line, `if all_files[i].endswith('.csv'):`, and under it, indented twice, `csv_files.append(all_files[i])`.

In [ ]:
csv_files = []

for i in range(len(all_files)):
    # Keep this name only if it ends in .csv
    --
        --

print(len(csv_files), 'CSV files')
print(csv_files)

<details><summary>Show the finished cell</summary>

```python
csv_files = []

for i in range(len(all_files)):
    # Keep this name only if it ends in .csv
    if all_files[i].endswith('.csv'):
        csv_files.append(all_files[i])

print(len(csv_files), 'CSV files')
print(csv_files)
```

</details>

24 names, the notes file gone. As with Lab 7's counter, keep `csv_files = []` in the **same cell** as the loop: if the loop sat in a cell of its own, every re-run would append the 24 names again.

## 8. Your turn: the loop over every file

This one is yours, not the instructor's. The loop below is sections 1 to 4 indented into a `for` loop over the indices of `csv_files`, so the analysis runs once per file. Four `--` to fill, all in lines you have already typed today, with `csv_files[i]` where DCP used to be:

1. this pass's file name: `csv_files[i]`, the single name for pass `i`, not the whole list
2. read it: `DATA_DIR / file_name`
3. the building code: the slice `file_name[6:9]`
4. the saved file's name, built from `building`, so each pass writes a different file

Two lines are new. `plt.close(fig)` after `savefig` throws the figure away once it is on disk; without it Colab keeps all 24 open, prints a warning at 20, and shows every one of them under the cell. The `print` at the end of each pass is how you watch the loop work.

In [ ]:
for i in range(len(csv_files)):
    # 1. this pass's file name, from the list
    file_name = --
    # 2. read it from DATA_DIR
    water_df = pd.read_csv(--)
    # 3. the building code from the file name
    building = --

    # Sections 2 and 3, unchanged, indented into the loop
    water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])
    water_df['Month'] = water_df['DateTime'].dt.month
    w_mean = np.mean(water_df[col])
    w_std = np.std(water_df[col], ddof=1)
    upper = w_mean + 3 * w_std
    lower = w_mean - 3 * w_std
    water_df = water_df[(water_df[col] < upper) & (water_df[col] > lower)]

    # Section 4, with a file name that changes with the building
    fig, ax = plt.subplots()
    water_df.boxplot(column=col, by='Month', ax=ax, grid=False)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(months)
    ax.set_xlabel('Month')
    ax.set_ylabel('Water consumption (gallons per day)')
    ax.set_title(building + ' water consumption by month')
    fig.suptitle('')
    # 4. one file per building, or every pass overwrites the last
    fig.savefig(FIGURES_DIR / --, dpi=150)
    plt.close(fig)
    print('saved', building, '-', len(water_df), 'days kept')

print(len(os.listdir(FIGURES_DIR)), 'figures in', FIGURES_DIR)

<details><summary>Show the finished cell</summary>

```python
for i in range(len(csv_files)):
    # 1. this pass's file name, from the list
    file_name = csv_files[i]
    # 2. read it from DATA_DIR
    water_df = pd.read_csv(DATA_DIR / file_name)
    # 3. the building code from the file name
    building = file_name[6:9]

    # Sections 2 and 3, unchanged, indented into the loop
    water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])
    water_df['Month'] = water_df['DateTime'].dt.month
    w_mean = np.mean(water_df[col])
    w_std = np.std(water_df[col], ddof=1)
    upper = w_mean + 3 * w_std
    lower = w_mean - 3 * w_std
    water_df = water_df[(water_df[col] < upper) & (water_df[col] > lower)]

    # Section 4, with a file name that changes with the building
    fig, ax = plt.subplots()
    water_df.boxplot(column=col, by='Month', ax=ax, grid=False)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(months)
    ax.set_xlabel('Month')
    ax.set_ylabel('Water consumption (gallons per day)')
    ax.set_title(building + ' water consumption by month')
    fig.suptitle('')
    # 4. one file per building, or every pass overwrites the last
    fig.savefig(FIGURES_DIR / (building + '_water_boxplot.png'), dpi=150)
    plt.close(fig)
    print('saved', building, '-', len(water_df), 'days kept')

print(len(os.listdir(FIGURES_DIR)), 'figures in', FIGURES_DIR)
```

</details>

## 9. The results

24 passes, 24 figures, one click. **Run as is:** the cell lists the saved files and shows one of them. Change `ARC` to any other code and run it again.

In [ ]:
from IPython.display import Image

print(sorted(os.listdir(FIGURES_DIR)))
Image(filename=str(FIGURES_DIR / 'ARC_water_boxplot.png'))

Every figure has the same axes, the same labels and the same cleaning, so the buildings can be compared: ARC uses the most water of the 24 and peaks in July; CDL, at the other end, uses a few dozen gallons a day. The loop did not print anything but a line per building, which is the point. In the activity the loop collects **numbers** from each file instead of a picture, and the appends of section 7 are how.

**Three things to remember about a loop over files**

1. **Each file must have the same layout**: the same column names in the same order with the same kind of data. The loop looks for `DateTime` and `Water ( Gallons )` in every file; a file whose column is called `Date` would stop it with a `KeyError`.
2. **Every output name must change with the input name**: the saved figure, a results file, a column in a table. A fixed name is overwritten on every pass, and only the last file's output survives.
3. **The loop variable is taken.** `i` indexes `csv_files`. A second loop inside the first one (over months, say) needs its own variable, `j`. And a dataframe variable like `water_df` holds only the **current** file: after the loop ends it holds the last file, EPS, and nothing else.

**Common mistakes**

| What you see | What went wrong |
| --- | --- |
| `FileNotFoundError: ... 'water_DCP.csv'` | `pd.read_csv(file_name)` without the folder. The file is in `DATA_DIR`, so it is `pd.read_csv(DATA_DIR / file_name)` |
| `TypeError: unsupported operand type(s) for /: 'PosixPath' and 'list'` | `DATA_DIR / csv_files`, the whole list, inside the loop. Pass `i` needs the one name, `csv_files[i]` |
| `KeyError: 'DateTime'` on one pass | The loop reached `water_notes.txt`. Build `csv_files` with the `.endswith('.csv')` filter and loop over that, not over `all_files` |
| One figure in `figures`, for EPS | The `savefig` name is the same on every pass. Build it from `building` |
| The title says `_DC` or `DCP.` | The slice is off by one. The code is characters 6, 7 and 8: `file_name[6:9]` |
| 24 figures printed under the cell and a warning about 20 open figures | `plt.close(fig)` is missing after `savefig` |
| Only EPS is analyzed, once | The analysis lines are not indented, so they sit after the loop and run on the last file only |
| `csv_files` has 48 names | `csv_files = []` sits in another cell and the loop cell was run twice. Keep the two together |
| `AttributeError: 'list' object has no attribute 'endswith'` | `all_files.endswith(...)` asks the whole list. The question is for one name, `all_files[i]` |

## 10. Why this matters

- **Module 1, again:** the Arduino logged one file per run. A loop like this one plots or summarizes a whole semester of runs at once.
- **Any multi-site dataset:** one file per sensor, well, building or year is the normal case. The same loop fits one regression per file (Lab 5) or evaluates one classifier per file (Lab 8), with a list collecting the number you care about from each pass.
- **The one-liner:** `sorted(DATA_DIR.glob('*.csv'))` lists the CSV files in one line, the way `(my_array > 5).sum()` counted in one line in Lab 7. Today's loop shows what that line hides.
- **Lab 7 and Lab 8:** `for i in range(len(...))`, an `if` inside a loop, a value accumulated per pass. Nothing about the loop changed; what it visits did.

Now open the **activity** notebook.

## Wrap-up

Nothing to submit for the tutorial. Keep it open: the activity uses the same loop on the same 24 files, collecting numbers instead of saving figures.